# DINOv3 — Chinee apple weed detection (Colab)

Self-contained Colab notebook for the `dinov3-colab` experiment in `weed-detection-experiments`.

What it does:
1. Clones the official [facebookresearch/dinov3](https://github.com/facebookresearch/dinov3) repo.
2. Loads DINOv3 weights from your Google Drive.
3. Extracts per-patch features and finds the foreground via PCA over patch tokens.
4. Launches a Gradio UI where you can drop an image and see a box around the dominant object.

**Before running:** Runtime → Change runtime type → **GPU** (T4 free is fine).

**One-time:** accept the DINOv3 license at https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/ and place the `.pth` in your Drive (default expected path is shown in step 2).

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Weights are read from Drive so you don't re-upload them each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust paths if your Drive layout differs.

In [ ]:
import os

DINOV3_REPO = '/content/dinov3'
WEIGHTS_PATH = '/content/drive/MyDrive/dinov3/weights/dinov3_vitb16_pretrain_lvd1689m.pth'
ARCH = 'dinov3_vitb16'

assert os.path.isfile(WEIGHTS_PATH), f'Weights not found at {WEIGHTS_PATH} — update WEIGHTS_PATH or copy the .pth into Drive.'
print('Weights OK:', WEIGHTS_PATH)

## 3. Clone DINOv3 and install dependencies

In [ ]:
if not os.path.isdir(DINOV3_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/dinov3.git {DINOV3_REPO}

%pip install -q pillow scipy scikit-learn torchmetrics 'gradio>=4.0'

## 4. Inference code

Same logic as `dinov3_detect.py`, inlined so the notebook is self-contained.

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy.ndimage import find_objects, label
from torchvision import transforms

PATCH = 16
IMG_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

model = torch.hub.load(DINOV3_REPO, ARCH, source='local', weights=WEIGHTS_PATH).to(DEVICE).eval()

_tx = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.inference_mode()
def cls_saliency(image):
    # Cosine similarity between [CLS] and each patch token.
    # Higher = more aligned with the image's global object representation.
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    out = model.forward_features(x)
    cls = out['x_norm_clstoken'][0]
    patches = out['x_norm_patchtokens'][0]
    sim = torch.nn.functional.cosine_similarity(patches, cls.unsqueeze(0), dim=-1)
    grid = IMG_SIZE // PATCH
    return sim.float().cpu().numpy().reshape(grid, grid)

def foreground_mask(sim):
    s = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
    return (s > s.mean()).astype(np.uint8)

def largest_component_bbox(mask):
    lbl, n = label(mask)
    if n == 0:
        return None
    sizes = np.bincount(lbl.ravel())
    sizes[0] = 0
    idx = int(sizes.argmax())
    sl = find_objects(lbl == idx)[0]
    return sl[1].start, sl[0].start, sl[1].stop, sl[0].stop

def annotate(image, bbox_grid, grid_size):
    if bbox_grid is None:
        return image
    W, H = image.size
    sx, sy = W / grid_size, H / grid_size
    x0, y0, x1, y1 = bbox_grid
    out = image.copy()
    ImageDraw.Draw(out).rectangle([x0 * sx, y0 * sy, x1 * sx, y1 * sy], outline='red', width=4)
    return out

def infer(image):
    sim = cls_saliency(image)
    bbox = largest_component_bbox(foreground_mask(sim))
    return annotate(image, bbox, sim.shape[0])

## 5. Debug — saliency heatmap

Upload an image to see what the model considers foreground. Useful when the Gradio output puts the box on the wrong region.

Panels: input | raw [CLS] saliency | overlay | mask + bbox.


In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

sim = cls_saliency(img)
mask = foreground_mask(sim)
bbox_grid = largest_component_bbox(mask)

sim_norm = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
sim_big = Image.fromarray((sim_norm * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)
mask_big = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size, Image.NEAREST)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(sim_big, cmap='viridis'); axes[1].set_title('[CLS] saliency'); axes[1].axis('off')
axes[2].imshow(img); axes[2].imshow(sim_big, cmap='viridis', alpha=0.5); axes[2].set_title('Overlay'); axes[2].axis('off')
axes[3].imshow(img); axes[3].imshow(mask_big, cmap='Reds', alpha=0.4)
if bbox_grid is not None:
    W, H = img.size
    sx, sy = W / sim.shape[0], H / sim.shape[0]
    x0, y0, x1, y1 = bbox_grid
    axes[3].add_patch(plt.Rectangle((x0 * sx, y0 * sy), (x1 - x0) * sx, (y1 - y0) * sy, fill=False, edgecolor='red', linewidth=3))
axes[3].set_title('Mask + bbox'); axes[3].axis('off')
plt.tight_layout(); plt.show()

print(f'Saliency  min={sim.min():.3f}  max={sim.max():.3f}  mean={sim.mean():.3f}')
print(f'Mask     coverage={mask.mean() * 100:.1f}% of patches  ({mask.sum()} of {mask.size})')
print(f'Bbox     {bbox_grid}  (in patch grid; image is {img.size})')


## 6. Launch the Gradio UI

Click the public `*.gradio.live` URL to open the interface in a new tab.

In [ ]:
import gradio as gr

gr.Interface(
    fn=infer,
    inputs=gr.Image(type='pil', label='Upload'),
    outputs=gr.Image(type='pil', label='Detected object'),
    title='DINOv3 object localization — Chinee apple',
    description='Foreground is found via PCA over DINOv3 patch tokens; the largest connected blob is boxed. Backbone-only — no class label.',
).launch(share=True)